# A1 — granule subtypes and subtype density, SSAM only

mcDETECT's granule-subtyping and subtype-density chain, applied to the SSAM detections.

**Why SSAM only.** `A1_filter_de.ipynb` §4 renders a subtype heatmap for all ten arms. Baysor's
clusters have no block structure — the same markers are elevated across nearly every cluster, so no
cluster corresponds to a compartment and a manual cluster → compartment mapping cannot be made.
SSAM's do resolve, with distinct Camk2a, Vamp2, Ddn, Shank1, Slc17a7 and Syp/Syt1 blocks. Subtyping
is therefore meaningful for SSAM and not for Baysor, and this notebook covers SSAM's five
populations: `all` × {pop1, pop2, pop3} and `markers` × {pop1, pop2}.

**Scope stops at subtype density.** Neuropil microdomains and microdomain DE/GSEA are deliberately
*not* here: `A1_filter_de.ipynb` §5 already does that against mcDETECT's own published microdomains,
which is the better-controlled comparison. Nothing in this notebook should duplicate it.

**The subtyping loop is mcDETECT's, in three steps** (`benchmark_subtyping.ipynb` cell 22):

1. **§2** cluster on the 34 markers and draw the heatmap — `heatmap_subtype.jpeg`;
2. **§3** read that panel and assign each cluster to a compartment by hand;
3. **§4** redraw the same heatmap with the clusters grouped by compartment —
   `heatmap_subtype_ordered.jpeg` — which is how you check the assignment holds.

§5 then computes subtype density and §6 correlates the synaptic detections against the
genetic-labeling reference. Steps 2–4 loop: if the ordered panel shows a cluster sitting in the
wrong block, edit §3 and re-run §3–§4. Only §5–§6 have to follow.

**§6 is the one part that can fail against outside data.** Everything before it is internal — it
describes the detections and their distribution, but nothing in it could be contradicted by a
measurement. §6 asks whether the *synaptic* detections are spread over brain regions the way
synapses are, against volume-EM + genetic-labeling densities (Santuy et al. 2020), reproducing
mcDETECT's own check. WT only, since the reference is a WT measurement.

**Clustering is done from scratch here**, with the seed set below — the labels are *not* read back
from `A1_filter_de.ipynb` §4. That keeps this analysis self-contained, but it also means the cluster
ids below belong to *this* notebook's run and the manual mapping in §3 is only valid for the seed it
was written against.

Reads `A1_filter_de.ipynb`'s caches (`<config>_features.parquet`, `<config>_profile.h5ad`), so run
that notebook's §2–§3 first. Outputs land in `../output/postproc/ssam_subtypes/`; `A1_figures.R` §6
plots the §5 density tables. §6 here is tables only — mcDETECT reports it as two numbers and so do
we, so nothing on the R side reads it.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

import anndata
import numpy as np
import pandas as pd
import scanpy as sc

sys.path.insert(0, str(Path.cwd()))          # run this notebook from postproc/
import postproc_config as C
import sphere_features as SF

warnings.filterwarnings("ignore")
sc.settings.verbosity = 1

# ================================ knobs ================================ #
METHOD = "ssam"          # this notebook is SSAM-only by design; see the header

# Nothing here is cached: clustering five SSAM arms takes a couple of minutes and the density step
# seconds, and the prescribed workflow is "run with the mappings empty, fill them in, re-run" --
# which any output-exists check would defeat by skipping exactly the second pass.

# Clustering seed, PER POPULATION -- different arms may need different seeds to give a clean
# separation. Cluster ids are seed-dependent, so changing one arm's seed reshuffles that arm's ids
# and invalidates ONLY that arm's mapping in section 3; the other four are untouched. A stale
# mapping assigns the wrong compartments without failing, so re-read the heatmap after any change.
SUBTYPE_SEED = {
    ("all", "pop1"):     C.SUBTYPE_SEED,     # mcDETECT's published value is 1
    ("all", "pop2"):     C.SUBTYPE_SEED,
    ("all", "pop3"):     0,
    ("markers", "pop1"): 25,
    ("markers", "pop2"): C.SUBTYPE_SEED,
}

DENSITY_SEED = 0         # bootstrap CI only; mcDETECT's is unseeded

ARMS = [(g, p) for g in C.GENESETS for p in C.populations(g)]
OUT_ROOT = C.POSTPROC_DIR / "ssam_subtypes"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

def arm_dir(geneset, population):
    d = OUT_ROOT / f"{METHOD}_{geneset}_{population}"
    d.mkdir(parents=True, exist_ok=True)
    return d

print("method      :", METHOD)
print("arms        :", [f"{g}/{p}" for g, p in ARMS])
print("k           :", C.K_SUBTYPE, "| seeds:", {f"{g}/{p}": SUBTYPE_SEED[(g, p)] for g, p in ARMS})
print("output      :", OUT_ROOT)


/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/Users/chenyang/miniconda3/envs/mcDETECT-env/lib/python3.10/site-p

method      : ssam
arms        : ['all/pop1', 'all/pop2', 'all/pop3', 'markers/pop1', 'markers/pop2']
k           : 15 | seeds: {'all/pop1': 1, 'all/pop2': 1, 'all/pop3': 0, 'markers/pop1': 25, 'markers/pop2': 1}
output      : /Users/chenyang/Desktop/mcDETECT/R2_revision/baysor_ssam_merscope/output/postproc/ssam_subtypes


## 1. Load the SSAM detections

Both samples are pooled per arm, exactly as mcDETECT clusters WT and AD granules together.

No subsampling is applied: §2 passes `max_cells=None`, so the labels cover the complete population,
which is what makes the density downstream valid. That is affordable because SSAM's arms are small
— the largest is `markers`/`pop1` at ~175k detections, against Baysor's ~6M — and §2 asserts the
label count matches, so a future arm that outgrows this cannot misalign silently.


In [2]:
arms = {}
for geneset, population in ARMS:
    parts = []
    for sample in C.SAMPLES:
        f_path = C.features_path(METHOD, sample, geneset)
        p_path = C.profile_path(METHOD, sample, geneset)
        if not (f_path.exists() and p_path.exists()):
            raise FileNotFoundError(f"{f_path.name} / {p_path.name} missing -- "
                                    f"run A1_filter_de.ipynb sections 2-3 first.")
        feats = pd.read_parquet(f_path, columns=[population, "sphere_x", "sphere_y"])
        adata = anndata.read_h5ad(p_path)
        keep = feats[population].to_numpy().astype(bool)
        sub = adata[keep].copy()
        sub.obs["sample"] = sample
        sub.obs["sphere_x"] = feats.loc[keep, "sphere_x"].to_numpy()
        sub.obs["sphere_y"] = feats.loc[keep, "sphere_y"].to_numpy()
        parts.append(sub)
        del adata

    arm = anndata.concat(parts, index_unique="-")
    arm.obs["granule_id"] = [f"gnl_{i}" for i in range(arm.n_obs)]
    arms[(geneset, population)] = arm
    del parts
    print(f"[{geneset}/{population}] {arm.n_obs:,} detections "
          f"({dict(arm.obs['sample'].value_counts())})")


[all/pop1] 164,890 detections ({'WT': 88517, 'AD': 76373})
[all/pop2] 94,194 detections ({'WT': 49785, 'AD': 44409})
[all/pop3] 40,307 detections ({'WT': 21713, 'AD': 18594})
[markers/pop1] 175,250 detections ({'WT': 92687, 'AD': 82563})
[markers/pop2] 115,931 detections ({'WT': 61163, 'AD': 54768})


## 2. Cluster from scratch, and render the heatmap you subtype from

mcDETECT's procedure unchanged (`code/benchmark/benchmark_subtyping.ipynb`,
`run_manual_subtyping()` in cell 15 and the heatmap in cell 22): normalise on the **full 290-gene
panel**, then subset to the 34 `REF_GENES`; `MiniBatchKMeans(15, batch_size=5000, n_init=20)` on
that marker matrix — no PCA, no plain KMeans; then `sc.pl.heatmap(cmap="Reds",
standard_scale="var", swap_axes=True, dendrogram=False)` at dpi 500. Panels are therefore directly
comparable to `output/MERSCOPE_WT_AD_comparison/heatmap_subtype.jpeg`.

Alongside each heatmap this writes `subtype_marker_means.csv`, the cluster × marker mean table.
Reading the mapping off that table is far less error-prone than eyeballing the figure, and it is
what the next section expects you to use.

`run_info.csv` records the seed, k and cluster sizes per arm, so a mapping written under one seed
can be caught if the seed later changes.


In [3]:
labels = {}
clustered = {}    # the marker-subset AnnData per arm, kept so a heatmap can be redrawn (title,
                  # figure size) without paying for the clustering again
run_info = []

for (geneset, population), arm in arms.items():
    d = arm_dir(geneset, population)
    seed = SUBTYPE_SEED[(geneset, population)]
    tag = f"{geneset}/{population}"
    print(f"[{tag}] seed = {seed}")

    # max_cells=None -> no subsampling, so the labels align 1:1 with `arm` and can be attached
    # back to it in section 3. Every SSAM arm is small enough that this is free.
    sub, lab, n_used = SF.subtype_cluster(arm, seed=seed, max_cells=None)
    assert len(lab) == arm.n_obs, (
        f"{tag}: got {len(lab)} labels for {arm.n_obs} detections -- the labels would misalign")
    labels[(geneset, population)] = lab
    clustered[(geneset, population)] = sub

    # render from the clustering just done, rather than re-clustering inside subtype_heatmap
    SF.render_subtype_heatmap(sub, d / "heatmap_subtype.jpeg",
                              marker_means_path=d / "subtype_marker_means.csv",
                              title=f"SSAM | {geneset} | {population}  "
                                    f"(n = {arm.n_obs:,}; seed {seed})",
                              verbose=False)

    sizes = lab.value_counts().sort_index()
    run_info.append({"method": METHOD, "geneset": geneset, "population": population,
                     "seed": seed, "k": C.K_SUBTYPE,
                     "n_detections": int(arm.n_obs), "n_clustered": int(n_used),
                     **{f"n_cluster_{i}": int(sizes.get(str(i), 0)) for i in range(C.K_SUBTYPE)}})
    print(f"    cluster sizes: {sizes.to_dict()}")

run_info = pd.DataFrame(run_info)
run_info.to_csv(OUT_ROOT / "run_info.csv", index=False)
print("\nwrote run_info.csv -- the seed recorded there is the one each mapping in section 3 belongs to")
run_info[["geneset", "population", "seed", "k", "n_detections", "n_clustered"]]


[all/pop1] seed = 1
    clustered 164,890 detections on 34/34 markers, k = 15, seed = 1
    cluster sizes: {'0': 5518, '1': 37593, '2': 9096, '3': 9285, '4': 15357, '5': 6670, '6': 5951, '7': 10919, '8': 6070, '9': 20242, '10': 8513, '11': 9948, '12': 6578, '13': 7000, '14': 6150}
[all/pop2] seed = 1
    clustered 94,194 detections on 34/34 markers, k = 15, seed = 1
    cluster sizes: {'0': 9154, '1': 5022, '2': 3632, '3': 5343, '4': 2653, '5': 15896, '6': 8649, '7': 6080, '8': 3936, '9': 3599, '10': 10385, '11': 7620, '12': 1976, '13': 7171, '14': 3078}
[all/pop3] seed = 0
    clustered 40,307 detections on 34/34 markers, k = 15, seed = 0
    cluster sizes: {'0': 1880, '1': 1656, '2': 2223, '3': 3251, '4': 1981, '5': 1589, '6': 5533, '7': 2210, '8': 3321, '9': 5066, '10': 1991, '11': 3269, '12': 1958, '13': 2157, '14': 2222}
[markers/pop1] seed = 25
    clustered 175,250 detections on 34/34 markers, k = 15, seed = 25
    cluster sizes: {'0': 21211, '1': 20217, '2': 7139, '3': 14086, '

,geneset,population,seed,k,n_detections,n_clustered
0,all,pop1,1,15,164890,164890
1,all,pop2,1,15,94194,94194
2,all,pop3,0,15,40307,40307
3,markers,pop1,25,15,175250,175250
4,markers,pop2,1,15,115931,115931


### What defines each cluster — read the mapping off this

`subtype_marker_means.csv` holds the raw cluster × marker means, but raw means are dominated by
whichever markers are simply abundant. The heatmap you are looking at is column-scaled
(`standard_scale="var"`, each marker scaled to [0, 1] across clusters), so this prints the same view
as a table: per cluster, the markers whose scaled mean is at least `TOP_MARKER_THR`, strongest
first.

That is the evidence for the mapping in section 3; the compartment call itself is yours.

**On marker groupings.** mcDETECT does *not* define a gene → compartment map — it assigns whole
*clusters* to compartments by eye, and its only gene-level grouping is `SYNAPTIC_SUBTYPES`, which
operates on subtype names rather than genes. The grouping below is therefore an indicative reading
aid based on what the 34 `REF_GENES` are, not something inherited from mcDETECT, and it is offered
only to orient the eye:

| | markers |
|---|---|
| pre-synaptic | Bsn, Syn1, Syp, Syt1, Vamp2, Cplx2, Stx1a, Nrxn1, Slc17a6, Slc17a7, Slc32a1 |
| post-synaptic | Dlg3, Dlg4, Gphn, Gria1, Gria2, Homer1, Homer2, Nlgn1, Nlgn2, Nlgn3, Shank1, Shank3, Camk2a |
| dendritic | Map1a, Map2, Ddn, Cyfip2 |
| axonal | Ank3, Nav1, Nfasc, Mapt, Tubb3, Gap43 |

All 34 markers are accounted for. Several are not exclusive to one compartment — `Camk2a` and
`Map2` in particular are widely distributed — so treat the table as a prior, not a rule.


In [4]:
TOP_MARKER_THR = 0.5     # scaled mean at or above this counts as "defines the cluster"
TOP_MARKER_MAX = 6       # at most this many listed per cluster, strongest first

for geneset, population in ARMS:
    d = arm_dir(geneset, population)
    means = pd.read_csv(d / "subtype_marker_means.csv", index_col=0)
    means.index = means.index.astype(str)
    # column-scale each marker across clusters, exactly as standard_scale="var" does for the figure
    scaled = (means - means.min()) / (means.max() - means.min()).replace(0, np.nan)
    sizes = labels[(geneset, population)].value_counts()

    print(f"\n[{geneset}/{population}]")
    for cl in sorted(means.index, key=int):
        row = scaled.loc[cl].dropna().sort_values(ascending=False)
        hits = row[row >= TOP_MARKER_THR]
        if hits.empty:
            hits = row.head(2)
        shown = hits.head(TOP_MARKER_MAX)
        extra = f"  (+{len(hits) - len(shown)} more)" if len(hits) > len(shown) else ""
        desc = ", ".join(f"{g} {v:.2f}" for g, v in shown.items())
        print(f"   cluster {cl:>2}  (n = {int(sizes.get(cl, 0)):>7,})  {desc}{extra}")

print(f"\nA cluster listing many markers at ~1.00 is one that is the maximum for most of them --"
      f"\nusually a sparse cluster rather than a compartment. Check its n and the heatmap before"
      f"\nassigning it; leaving it unlisted is a legitimate outcome.")



[all/pop1]
   cluster  0  (n =   5,518)  Bsn 1.00, Map1a 1.00, Camk2a 0.82, Ank3 0.82, Shank3 0.73, Slc17a7 0.68  (+8 more)
   cluster  1  (n =  37,593)  Nfasc 0.61, Gphn 0.54
   cluster  2  (n =   9,096)  Shank1 1.00, Homer2 0.98, Camk2a 0.97, Map2 0.87, Shank3 0.85, Dlg4 0.75
   cluster  3  (n =   9,285)  Shank1 1.00, Homer2 1.00, Shank3 1.00, Map2 1.00, Dlg4 1.00, Ddn 0.97  (+1 more)
   cluster  4  (n =  15,357)  Camk2a 1.00, Map2 0.68, Dlg4 0.64, Shank3 0.61, Homer2 0.59, Nrxn1 0.55
   cluster  5  (n =   6,670)  Ank3 1.00, Nlgn2 1.00, Nfasc 1.00, Nlgn3 0.79, Nav1 0.78, Gphn 0.77  (+7 more)
   cluster  6  (n =   5,951)  Slc17a7 1.00, Cyfip2 0.99, Stx1a 0.98, Nrxn1 0.85, Homer1 0.81, Dlg3 0.81  (+4 more)
   cluster  7  (n =  10,919)  Camk2a 1.00, Ddn 0.99, Dlg4 0.96, Map2 0.89, Shank3 0.80, Homer2 0.63
   cluster  8  (n =   6,070)  Tubb3 1.00, Syn1 1.00, Gap43 1.00, Dlg3 1.00, Cyfip2 1.00, Syt1 1.00  (+13 more)
   cluster  9  (n =  20,242)  Map1a 0.91, Ank3 0.83, Nfasc 0.79, Nrxn1 0

## 3. Manual subtyping — **fill this in from section 2**

mcDETECT's format (`apply_manual_annotation`): a dict of **subtype → list of cluster ids**, where
keys joined with `" & "` denote clusters expressing more than one compartment. Those combination
keys collapse to `mixed`; clusters left unlisted stay unannotated, which is allowed — they are
counted in the `overall` density row but in no per-subtype row.

**One mapping per population**, because each was clustered separately and the ids do not correspond
across them. Read each from that arm's `subtype_marker_means.csv` / `heatmap_subtype.jpeg`.

On the **first pass** leave them empty and just run: the notebook goes end to end, §4 skips, and
the density table comes out with `overall` rows only. Then fill the mappings and re-run from here —
§4 will redraw each heatmap grouped by compartment so you can check the call before trusting §5.

For reference, mcDETECT's own seed-1 assignment on its granules was `pre-syn ["0","11","12","13"]`,
`post-syn ["1","2","3"]`, `dendrites ["4","8"]`, `pre & post ["6"]`, `post & den ["5","9","10"]`,
`pre & post & den ["7","14"]`, with `axons` and `others` empty.

Cluster ids may be written as integers or strings — `SF.apply_manual_subtypes` coerces them. It
raises rather than silently producing an all-`NaN` column if an id is out of range, if an id is
listed under two subtypes, or if a **key** is misspelt: mcDETECT casts the collapsed column to a
Categorical over exactly `pre-syn / post-syn / dendrites / axons / mixed / others`, so `"presyn"`
or `"pre-synaptic"` would otherwise drop that whole cluster to `NaN` without a word.

⚠️ Each mapping is valid only for **that population's** seed in `SUBTYPE_SEED` (first cell). The
seeds are per population, so changing one arm's seed invalidates only that arm's block — the other
four keep their mappings. `run_info.csv` records which seed each arm was clustered under.


In [5]:
# ============================ EDIT ME ============================ #
# One mapping per population. Each was clustered separately, so cluster id 3 in `all/pop1` has
# nothing to do with cluster id 3 in `markers/pop2` -- fill each block from its own arm.
#
# Put cluster ids (0-14) in the lists. Ints or strings both work. Leave a subtype empty if no
# cluster matches it; leave a cluster out entirely if it matches nothing -- those detections are
# counted in the `overall` density row but in no per-subtype row.
#
# Keys containing " & " collapse to "mixed" in the density table, as in mcDETECT.
# Every block is written out in full on purpose: a shared template copied five times would alias
# the same nine lists across all five populations, so editing one would silently edit them all.
# ================================================================= #

MANUAL_SUBTYPE_MAPPING = {
    # --- SSAM / all / pop1 --- read from ssam_all_pop1/ in section 2
    ("all", "pop1"): {
        "pre-syn":          ["10", "11", "13"],
        "post-syn":         ["2", "4", "5"],
        "dendrites":        ["12"],
        "axons":            [],
        "pre & post":       ["14"],
        "pre & den":        ["6"],
        "post & den":       ["3", "7", "9"],
        "pre & post & den": ["0", "8"],
        "others":           ["1"],
    },
    # --- SSAM / all / pop2 --- read from ssam_all_pop2/ in section 2
    ("all", "pop2"): {
        "pre-syn":          ["3", "13"],
        "post-syn":         ["1", "6", "11", "12"],
        "dendrites":        ["0", "7"],
        "axons":            [],
        "pre & post":       [],
        "pre & den":        [],
        "post & den":       ["4", "9", "10"],
        "pre & post & den": ["2", "8", "14"],
        "others":           ["5"],
    },
    # --- SSAM / all / pop3 --- read from ssam_all_pop3/ in section 2
    ("all", "pop3"): {
        "pre-syn":          ["1", "13"],
        "post-syn":         ["3", "6", "8"],
        "dendrites":        ["7", "11"],
        "axons":            [],
        "pre & post":       ["0", "4"],
        "pre & den":        ["5"],
        "post & den":       ["9", "14"],
        "pre & post & den": ["2", "10"],
        "others":           ["12"],
    },
    # --- SSAM / markers / pop1 --- read from ssam_markers_pop1/ in section 2
    ("markers", "pop1"): {
        "pre-syn":          ["6", "14"],
        "post-syn":         ["3", "5", "7", "8"],
        "dendrites":        ["2", "11", "12"],
        "axons":            [],
        "pre & post":       [],
        "pre & den":        ["13"],
        "post & den":       ["0", "4", "9"],
        "pre & post & den": ["10"],
        "others":           ["1"],
    },
    # --- SSAM / markers / pop2 --- read from ssam_markers_pop2/ in section 2
    ("markers", "pop2"): {
        "pre-syn":          ["3", "12"],
        "post-syn":         ["2", "10"],
        "dendrites":        ["4", "6"],
        "axons":            [],
        "pre & post":       ["8"],
        "pre & den":        ["1", "13"],
        "post & den":       ["0", "5", "7", "14"],
        "pre & post & den": ["9"],
        "others":           ["11"],
    },
}

# ---- apply, and persist ---- #
for (geneset, population), arm in arms.items():
    mapping = MANUAL_SUBTYPE_MAPPING[(geneset, population)]
    fine, simple = SF.apply_manual_subtypes(labels[(geneset, population)], mapping)
    arm.obs["granule_subtype_kmeans"] = labels[(geneset, population)].to_numpy()
    arm.obs["granule_subtype_manual"] = fine
    arm.obs["granule_subtype_manual_simple"] = simple

    n_unassigned = int(pd.isna(simple).sum())
    tag = f"{geneset}/{population}"
    if n_unassigned == arm.n_obs:
        print(f"[{tag}] mapping is empty -- density will carry the 'overall' row only")
    elif n_unassigned:
        print(f"[{tag}] {n_unassigned:,} of {arm.n_obs:,} detections in unlisted clusters "
              f"(subtype = NaN; counted in 'overall' only)")
    else:
        print(f"[{tag}] all {arm.n_obs:,} detections assigned")

    # mcDETECT's label schema carries the coordinates too; sphere_x/sphere_y are this pipeline's
    # per-sample raw frame -- the one the density step uses -- not the registered global_x/global_y
    out = arm.obs[["sample", "granule_id", "granule_subtype_kmeans",
                   "granule_subtype_manual", "granule_subtype_manual_simple",
                   "sphere_x", "sphere_y"]].copy()
    out["seed"] = SUBTYPE_SEED[(geneset, population)]
    out.to_parquet(arm_dir(geneset, population) / "granule_subtype_labels.parquet", index=False)

pd.DataFrame({f"{g}/{p}": arms[(g, p)].obs["granule_subtype_manual_simple"]
              .value_counts(dropna=False) for g, p in ARMS}).fillna(0).astype(int)


[all/pop1] all 164,890 detections assigned
[all/pop2] all 94,194 detections assigned
[all/pop3] all 40,307 detections assigned
[markers/pop1] all 175,250 detections assigned
[markers/pop2] all 115,931 detections assigned


,all/pop1,all/pop2,all/pop3,markers/pop1,markers/pop2
granule_subtype_manual_simple,,,,,
pre-syn,25461,12514,3813,19784,11665
post-syn,31123,23267,12105,46855,23860
dendrites,6578,15234,5479,28336,15706
axons,0,0,0,0,0
mixed,64135,27283,16952,60058,55305
others,37593,15896,1958,20217,9395


## 4. Ordered heatmap — does the mapping hold up?

The third step of mcDETECT's subtyping loop, and the check on §3: redraw the **same heatmap** with
the clusters reordered so each compartment forms one contiguous block
(`benchmark_subtyping.ipynb` cell 22 → `heatmap_subtype_ordered.jpeg`). Blocks run in mcDETECT's
subtype order — `pre-syn`, `post-syn`, `dendrites`, `axons`, `mixed`, `others` — each sorted
numerically inside, at mcDETECT's `figsize=(10, 6.15)`.

Read it as a verdict on the mapping, not as a new result. If a compartment's block is coherent —
its markers elevated across the whole block and quiet elsewhere — the call is supported. If one
cluster inside a block looks like its neighbours in a *different* block, move it and re-run this
cell; nothing downstream has to be recomputed, because §3 already wrote the labels.

Colours carry over from the unordered panel, so a cluster is the same colour in both. This draws
from `clustered[...]` — the object §2 already clustered — so no arm is clustered twice.

**Clusters left unlisted in §3 are appended at the end**, in numeric order, rather than dropped.
mcDETECT's published mapping assigns all 15 clusters so it never faced the question; dropping them
would make this panel hold fewer detections than the unordered one it is meant to be compared
against. `cluster_subtype_map.csv` records the block each cluster landed in.

On the first pass, with the mappings still empty, this cell skips — the ordering would be the
identity and the panel a duplicate.


In [6]:
for geneset, population in ARMS:
    d = arm_dir(geneset, population)
    tag = f"{geneset}/{population}"
    print(f"[{tag}]")
    SF.render_ordered_subtype_heatmap(
        clustered[(geneset, population)],
        MANUAL_SUBTYPE_MAPPING[(geneset, population)],
        d / "heatmap_subtype_ordered.jpeg",
        cluster_map_path=d / "cluster_subtype_map.csv",
        title=f"SSAM | {geneset} | {population}  (ordered by subtype; "
              f"n = {arms[(geneset, population)].n_obs:,}; "
              f"seed {SUBTYPE_SEED[(geneset, population)]})")


[all/pop1]
    wrote heatmap_subtype_ordered.jpeg  [pre-syn:3, post-syn:3, dendrites:1, mixed:7, others:1]
[all/pop2]
    wrote heatmap_subtype_ordered.jpeg  [pre-syn:2, post-syn:4, dendrites:2, mixed:6, others:1]
[all/pop3]
    wrote heatmap_subtype_ordered.jpeg  [pre-syn:2, post-syn:3, dendrites:2, mixed:7, others:1]
[markers/pop1]
    wrote heatmap_subtype_ordered.jpeg  [pre-syn:2, post-syn:4, dendrites:3, mixed:5, others:1]
[markers/pop2]
    wrote heatmap_subtype_ordered.jpeg  [pre-syn:2, post-syn:2, dendrites:2, mixed:8, others:1]


## 5. Granule-subtype density per brain region, WT vs AD

Reproduces mcDETECT's published subtype-density analysis
(`code/benchmark/benchmark_subtyping.ipynb` cells 19 and 22) on these detections, so the two are
directly comparable:

* density per `(brain_area, subtype)` = detections in that area / spots in that area, on the
  sample's **own** 50 µm spot grid — a mean per-spot count, not a count per unit area — plus an
  `overall` row per area covering **all** detections including those left unassigned in §3;
* **no capture-efficiency correction** — `CAPTURE_EFFICIENCY_COEF` is deliberately `1.0` here, so
  the AD counts are raw. mcDETECT divides its AD side by `0.818691`; at 1.22× that is essentially
  the whole of SSAM's apparent AD increase (see `postproc_config.py`), so it is not carried over;
* per-spot `density_sd` / `density_sem`, and a 500-resample bootstrap 95% CI (seeded here;
  mcDETECT's is not);
* WT-vs-AD `ttest_ind` on `log1p` per-spot counts, Bonferroni **within subtype** plus BH-FDR across
  all (area × subtype) tests, and the star strings from `mcDETECT.utils.p_val_to_star`.

The 16 output columns match `subtype_density_per_region_granule_adata_tsne.csv` exactly, so
`A1_figures.R` §6 draws the bars with mcDETECT's own code path. **The WT densities are directly
comparable to mcDETECT's published table; the AD densities are not on the same scale**, because
of the correction bullet above — state that wherever the two are put side by side.

**Coordinates:** `sphere_x` / `sphere_y`, each sample's own raw frame, against that sample's own
`spots.h5ad`. Brain areas are enumerated from the spot object, `Unknown` included, as mcDETECT does;
the R side filters to the nine plotted regions.


In [7]:
spots_grid = {s: anndata.read_h5ad(C.spots_path(s)) for s in C.SAMPLES}
print("spot grids:", {s: v.n_obs for s, v in spots_grid.items()})

density_index = []
for geneset, population in ARMS:
    arm = arms[(geneset, population)]
    d = arm_dir(geneset, population)
    out_path = d / "subtype_density_per_region.csv"
    tag = f"{geneset}/{population}"

    # Deliberately NOT cached. The workflow is "run with the mappings empty, fill them in, re-run",
    # and a cache keyed on the file existing would skip exactly that second pass and leave the
    # overall-only table in place. Density on these arms takes seconds.
    dens_parts, spot_parts = [], []
    for sample in C.SAMPLES:
        m = (arm.obs["sample"] == sample).to_numpy()
        dd, ps = SF.subtype_density_per_region(
            det_x=arm.obs.loc[m, "sphere_x"].to_numpy(),
            det_y=arm.obs.loc[m, "sphere_y"].to_numpy(),
            det_subtype=arm.obs.loc[m, "granule_subtype_manual_simple"].to_numpy(),
            spots=spots_grid[sample], sample=sample,
            # a no-op while C.CAPTURE_EFFICIENCY_COEF == 1.0; kept so restoring mcDETECT's
            # correction is a one-constant change (see the markdown above)
            apply_capture_coef=(sample == "AD"), seed=DENSITY_SEED)
        dens_parts.append(dd)
        spot_parts.append(ps)

    out = SF.add_density_significance(
        pd.concat(dens_parts, ignore_index=True),
        pd.concat(spot_parts, ignore_index=True),
        setting=f"{METHOD}_{geneset}_{population}")
    out.to_csv(out_path, index=False)
    density_index.append({"geneset": geneset, "population": population, "path": str(out_path)})
    print(f"[{tag}] wrote {out.shape[0]} rows | subtypes: "
          f"{sorted(out.subtype.unique())}")

del spots_grid
pd.DataFrame(density_index)[["geneset", "population"]]


spot grids: {'WT': 17667, 'AD': 12604}
[all/pop1] wrote 120 rows | subtypes: ['dendrites', 'mixed', 'others', 'overall', 'post-syn', 'pre-syn']
[all/pop2] wrote 120 rows | subtypes: ['dendrites', 'mixed', 'others', 'overall', 'post-syn', 'pre-syn']
[all/pop3] wrote 120 rows | subtypes: ['dendrites', 'mixed', 'others', 'overall', 'post-syn', 'pre-syn']
[markers/pop1] wrote 120 rows | subtypes: ['dendrites', 'mixed', 'others', 'overall', 'post-syn', 'pre-syn']
[markers/pop2] wrote 120 rows | subtypes: ['dendrites', 'mixed', 'others', 'overall', 'post-syn', 'pre-syn']


,geneset,population
0,all,pop1
1,all,pop2
2,all,pop3
3,markers,pop1
4,markers,pop2


## 6. Synaptic density vs the genetic-labeling reference — WT only

mcDETECT's one **external** check on the subtyping, reproduced here
(`benchmark_subtyping.ipynb` cell 22, the WT correlation block). Sections 2–5 are all internal:
they say what the clusters look like and how the detections are distributed, but nothing in them
could be wrong in a way the data would notice. This one can fail. It asks whether the detections
called *synaptic* are spread across brain regions the way synapses actually are — against synapse
densities measured in an age-matched WT mouse by volume electron microscopy plus genetic labeling
(Santuy et al. 2020; the reference behind the manuscript's Fig. S9, values in `C.GT_SYNAPSE_DENSITY`).

* **Synaptic = the fine label**, `granule_subtype_manual` in `pre-syn`, `post-syn`, `pre & post`.
  Not the collapsed column — `pre & den`, `post & den` and `pre & post & den` are excluded, exactly
  as mcDETECT excludes them.
* **WT only.** The reference is a WT measurement, so AD never enters and
  `CAPTURE_EFFICIENCY_COEF` plays no part in this section.
* Density is the same quantity as §5 — a mean per-spot count on the sample's own 50 µm grid.
* Of the nine regions, **HPF-SR** (no reference value) and **FT** (mcDETECT's own exclusion, though
  its value does exist) are dropped, leaving **7**. Both vectors are min-max scaled and correlated
  with `weighted_corr` / `weighted_spearmanr` from `mcDETECT.utils`, weighted by spots per region.

**Read it with its limits in view.** Seven points, weighted by spot count, so Isocortex (2495 spots)
and MB (2075) carry most of it — a coarse instrument, but it is mcDETECT's instrument and the point
is that the two arms are measured the same way. The helper reproduces mcDETECT's published
**0.9098 / 0.8784** exactly when fed mcDETECT's own published labels, so a difference here is a
difference in the detections, not in the code.

mcDETECT's own coefficients enter the summary table as a **cited** row, not a recomputed one: cell
22 prints them but never persists the per-region density vector behind them.


In [8]:
spots_wt = anndata.read_h5ad(C.spots_path(C.GT_SAMPLE))
print(f"{C.GT_SAMPLE} spot grid: {spots_wt.n_obs:,} spots | "
      f"{len(C.GT_AREAS_USED)} of {len(C.AREA_LIST)} regions enter the correlation "
      f"(dropped {', '.join(C.GT_DROP_AREAS)})\n")

gt_rows = []
for geneset, population in ARMS:
    arm = arms[(geneset, population)]
    d = arm_dir(geneset, population)
    tag = f"{geneset}/{population}"
    m = (arm.obs["sample"] == C.GT_SAMPLE).to_numpy()
    print(f"[{tag}]")

    per_area, summary = SF.synaptic_density_vs_reference(
        arm.obs.loc[m, "sphere_x"], arm.obs.loc[m, "sphere_y"],
        arm.obs.loc[m, "granule_subtype_manual"],       # the FINE label, as mcDETECT
        spots_wt, setting=f"{METHOD}_{geneset}_{population}")
    per_area.to_csv(d / "synaptic_density_vs_reference.csv", index=False)

    # which of the three synaptic labels this arm's mapping actually produced -- an empty
    # "pre & post" is a property of the manual call in section 3, not of the code
    present = [s for s in C.SYNAPTIC_SUBTYPES
               if (arm.obs.loc[m, "granule_subtype_manual"] == s).any()]
    gt_rows.append({"method": METHOD, "geneset": geneset, "population": population,
                    "source": "computed here", "synaptic_labels": " + ".join(present) or "none",
                    **summary})

# mcDETECT's own result, CITED (see the markdown above) -- never recomputed here
gt_rows.append({"method": "mcdetect", "geneset": "--", "population": "--",
                "setting": "granule_adata_tsne", "source": C.MCDETECT_GT_SOURCE,
                "synaptic_labels": " + ".join(C.SYNAPTIC_SUBTYPES),
                "n_areas": len(C.GT_AREAS_USED),
                "r_pearson": C.MCDETECT_GT_PEARSON, "r_spearman": C.MCDETECT_GT_SPEARMAN,
                "note": ""})

gt_summary = pd.DataFrame(gt_rows)[
    ["method", "geneset", "population", "setting", "n_detections", "n_synaptic", "synaptic_frac",
     "synaptic_labels", "n_areas", "r_pearson", "r_spearman", "r_pearson_unw", "r_spearman_unw",
     "note", "source"]]
gt_summary.to_csv(OUT_ROOT / "ground_truth_correlation.csv", index=False)
del spots_wt
print("\nwrote ground_truth_correlation.csv")
gt_summary[["method", "geneset", "population", "n_synaptic", "synaptic_labels",
            "r_pearson", "r_spearman"]]


WT spot grid: 17,667 spots | 7 of 9 regions enter the correlation (dropped HPF-SR, FT)

[all/pop1]
    [ssam_all_pop1] 35,730 synaptic of 88,517 (40.4%) | 7 regions (dropped HPF-SR, FT)
    [ssam_all_pop1] weighted Pearson = -0.0967, weighted Spearman = 0.2004
[all/pop2]
    [ssam_all_pop2] 19,557 synaptic of 49,785 (39.3%) | 7 regions (dropped HPF-SR, FT)
    [ssam_all_pop2] weighted Pearson = -0.5593, weighted Spearman = -0.8975
[all/pop3]
    [ssam_all_pop3] 10,416 synaptic of 21,713 (48.0%) | 7 regions (dropped HPF-SR, FT)
    [ssam_all_pop3] weighted Pearson = 0.8416, weighted Spearman = 0.4797
[markers/pop1]
    [ssam_markers_pop1] 36,572 synaptic of 92,687 (39.5%) | 7 regions (dropped HPF-SR, FT)
    [ssam_markers_pop1] weighted Pearson = -0.2538, weighted Spearman = -0.2770
[markers/pop2]
    [ssam_markers_pop2] 24,750 synaptic of 61,163 (40.5%) | 7 regions (dropped HPF-SR, FT)
    [ssam_markers_pop2] weighted Pearson = -0.2068, weighted Spearman = -0.4363

wrote ground_truth_c

,method,geneset,population,n_synaptic,synaptic_labels,r_pearson,r_spearman
0,ssam,all,pop1,35730.0,pre-syn + post-syn + pre & post,-0.096711,0.200431
1,ssam,all,pop2,19557.0,pre-syn + post-syn,-0.559311,-0.897494
2,ssam,all,pop3,10416.0,pre-syn + post-syn + pre & post,0.841577,0.479672
3,ssam,markers,pop1,36572.0,pre-syn + post-syn,-0.253834,-0.277042
4,ssam,markers,pop2,24750.0,pre-syn + post-syn + pre & post,-0.206831,-0.436316
5,mcdetect,--,--,NaN,pre-syn + post-syn + pre & post,0.909800,0.878400


## Outputs

Everything lands in `../output/postproc/ssam_subtypes/` (git-ignored):

| file | contents |
|---|---|
| `run_info.csv` | seed, k, detections and cluster sizes per arm — the record of which seed a mapping belongs to |
| `ssam_<geneset>_<pop>/heatmap_subtype.jpeg` | §2 — the panel you read the mapping off, mcDETECT's call |
| `ssam_<geneset>_<pop>/subtype_marker_means.csv` | cluster × marker means, the table to read it from |
| `ssam_<geneset>_<pop>/heatmap_subtype_ordered.jpeg` | §4 — the same panel with clusters grouped by compartment, mcDETECT's `heatmap_subtype_ordered.jpeg` |
| `ssam_<geneset>_<pop>/cluster_subtype_map.csv` | which block each cluster landed in, and its size — the record of the manual call |
| `ssam_<geneset>_<pop>/granule_subtype_labels.parquet` | cluster and compartment labels per detection, plus the seed |
| `ssam_<geneset>_<pop>/subtype_density_per_region.csv` | WT/AD subtype density, mcDETECT's exact 16-column schema |
| `ssam_<geneset>_<pop>/synaptic_density_vs_reference.csv` | §6 — per-region synaptic density beside the genetic-labeling reference; `used` marks the 7 regions that enter the correlation |
| `ground_truth_correlation.csv` | §6 — weighted Pearson/Spearman per arm, with mcDETECT's published pair as a cited row |

Then run `A1_figures.R` §6, which discovers every arm here and draws one bar panel per subtype in
mcDETECT's format, so they can sit beside
`output/MERSCOPE_WT_AD_comparison/granule_density_<subtype>.jpeg`. The §6 correlation stays a table
on both sides: mcDETECT reports it as two printed coefficients, and `ground_truth_correlation.csv`
is the same two coefficients per arm.

**Scope note.** This notebook stops at subtype density by design. Neuropil microdomains and
microdomain DE/GSEA are handled in `A1_filter_de.ipynb` §5 against mcDETECT's *published*
microdomains — the comparison that holds the spatial partition fixed across methods — and are
deliberately not recomputed here.

**Baysor is absent by design**, not by oversight: its detections do not resolve into
compartment-specific clusters (see `A1_filter_de.ipynb` §4 and the heatmaps under
`subtype_heatmaps/`), so a manual cluster → compartment mapping cannot honestly be made for it.
